# 🧠 Semantic Research Atlas — GPU Embeddings

Este notebook genera embeddings para el dataset completo usando la **GPU gratuita de Colab**.

### Instrucciones:
1. Ve a **Runtime → Change runtime type → T4 GPU**
2. Sube `records.parquet` (de `data/raw/`) a la carpeta de archivos de Colab
3. Ejecuta todas las celdas
4. Descarga `embeddings.npy` al terminar
5. Cópialo a `data/processed/embeddings.npy` en tu máquina local

In [ ]:
# ── Instalar dependencias ──
!pip install -q sentence-transformers pyarrow

In [ ]:
# ── Verificar GPU ──
import torch
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No hay GPU. Ve a Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Subir records.parquet ──
# Opción A: Upload manual
from google.colab import files
import os

if not os.path.exists('records.parquet'):
    print("Sube el archivo records.parquet (de data/raw/)")
    uploaded = files.upload()
else:
    print(f"records.parquet ya existe ({os.path.getsize('records.parquet') / 1e6:.1f} MB)")

In [ ]:
# ── Cargar datos ──
import pyarrow.parquet as pq
import pandas as pd

df = pq.read_table('records.parquet').to_pandas()
print(f"Total registros: {len(df):,}")
print(f"Fuentes: {df['source'].value_counts().to_dict()}")
print(f"Columnas: {list(df.columns)}")
df.head(3)

In [ ]:
# ── Generar embeddings con GPU ──
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODEL_NAME = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 256  # GPU puede manejar batches más grandes

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando: {device.upper()}")

model = SentenceTransformer(MODEL_NAME, device=device)

# Preparar textos
texts = (df['title'].fillna('') + '. ' + df['abstract'].fillna('')).tolist()
print(f"Textos a procesar: {len(texts):,}")

# Embed
start = time.time()
embeddings = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=False,
)
elapsed = time.time() - start

print(f"\n✅ Completado en {elapsed/60:.1f} minutos")
print(f"   Shape: {embeddings.shape}")
print(f"   Velocidad: {len(texts)/elapsed:.0f} docs/seg")

In [ ]:
# ── Guardar embeddings ──
np.save('embeddings.npy', embeddings)
print(f"Guardado: embeddings.npy ({os.path.getsize('embeddings.npy') / 1e6:.1f} MB)")

# También guardar records.parquet procesado (por si acaso)
df.to_parquet('records_processed.parquet', index=False)
print(f"Guardado: records_processed.parquet")

In [ ]:
# ── Descargar ──
print("Descargando embeddings.npy...")
print("Cuando termine, cópialo a: data/processed/embeddings.npy")
files.download('embeddings.npy')